# 01 — Datos, partición y auditoría

**Proyecto Final Integrador — NLP (TC3009C)**
Traducción Automática Neuronal Inglés → Español

En este notebook:
1. Descargamos el corpus Tatoeba/ManyThings (`spa.txt`) y guardamos una copia en Google Drive.
2. Ejecutamos **exactamente** el bloque obligatorio de filtrado y partición 80/10/10 y obtenemos el **SHA-256 Test Audit ID**.
3. Exploramos longitudes y contamos cuántas oraciones de cada partición son *medias* (8–18 palabras) o *largas* (> 18).
4. Construimos los vocabularios `eng_word2idx` y `spa_word2idx` **solo con `train_pairs`** y los guardamos en Drive para los notebooks 02–06.

Este notebook no necesita GPU (*Entorno de ejecución → CPU*).

## 0. Configuración

In [ ]:
# --- Configuración del entorno (misma celda en todos los notebooks) ---
REPO_URL = 'https://github.com/EmilianoRL/NLP_Proyecto.git'
REPO_DIR = 'NLP_Proyecto'

import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('common'):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
elif not os.path.exists('common') and os.path.exists('../common'):
    %cd ..
print('Directorio de trabajo:', os.getcwd())

In [ ]:
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# random seed (los splits NO dependen de esta semilla: usan random.Random(23) del protocolo)
seed = 1234
random.seed(seed)
np.random.seed(seed)

Montamos Google Drive y creamos la carpeta del proyecto (recomendación operativa del profesor).
Todo lo que deban reutilizar otros notebooks (corpus, vocabularios, checkpoints, figuras) se guarda ahí.

In [ ]:
from common import checkpoint as ckpt

SAVE_DIR = ckpt.setup_save_dir()          # /content/drive/MyDrive/Proyecto_NLP_NMT en Colab
DATA_DIR = os.path.join(SAVE_DIR, 'data')
FIG_DIR = os.path.join(SAVE_DIR, 'figures')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print('Carpeta del proyecto en Drive:', SAVE_DIR)

## 1. Descarga del corpus

Equivale a `!wget -q https://www.manythings.org/anki/spa-eng.zip && unzip -q -o spa-eng.zip` del enunciado.
La primera vez se descarga y se guarda una copia del zip en Drive; después siempre se usa esa copia.
Así **ambos integrantes trabajan con la misma versión del corpus** y obtienen el mismo Test Audit ID,
aunque manythings.org actualice su archivo.

In [ ]:
from common.data import download_corpus, file_sha256

corpus_path = download_corpus('.', cache_dir=DATA_DIR)
corpus_sha = file_sha256(corpus_path)
print('Huella del archivo spa.txt:', corpus_sha)

## 2. Protocolo obligatorio de filtrado y partición

Bloque copiado **sin modificaciones** del enunciado (Listing 1).

In [ ]:
import random, hashlib

# 1. Descarga directa en Colab / entorno Linux:
# !wget -q https://www.manythings.org/anki/spa-eng.zip && unzip -q -o spa-eng.zip

with open('spa.txt', 'r', encoding='utf-8') as f:
    raw_pairs = [line.strip().split('\t')[:2] for line in f if '\t' in line]

# 2. Filtrado de complejidad mínima (>= 8 palabras en inglés, <= 120 tokens)
filtered_pairs = [
    (en, es) for en, es in raw_pairs
    if 8 <= len(en.split()) <= 120 and len(es.split()) <= 120
]

# 3. Mezcla determinista con semilla fija (obligatoria para auditoría)
rng = random.Random(23)
rng.shuffle(filtered_pairs)

# 4. Partición estricta 80% Train / 10% Validation / 10% Test
n = len(filtered_pairs)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_pairs = filtered_pairs[:n_train]
val_pairs   = filtered_pairs[n_train:n_train + n_val]
test_pairs  = filtered_pairs[n_train + n_val:]

# 5. Huella de auditoría (debe reportarse en el notebook y en el PDF)
test_hash = hashlib.sha256("\n".join([p[0] for p in test_pairs]).encode()).hexdigest()[:12]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print(f"SHA-256 Test Audit ID: {test_hash}")

Los notebooks 02–06 no repiten el bloque a mano: llaman a `common.data.load_splits()`, que contiene la misma lógica.
Comprobamos aquí que ambas versiones producen **exactamente** las mismas particiones.

In [ ]:
from common.data import load_splits

tr, va, te, h = load_splits('spa.txt', verbose=False)
assert tr == train_pairs and va == val_pairs and te == test_pairs and h == test_hash
print('OK: load_splits() reproduce exactamente el protocolo. Test Audit ID =', h)

Guardamos la auditoría en Drive. Si ya existía (por ejemplo, la generó el otro integrante),
verificamos que coincida: si no coincide, **alguien está usando otra versión del corpus**.

In [ ]:
audit = {
    'n_raw_pairs': len(raw_pairs),
    'n_filtered_pairs': n,
    'n_train': len(train_pairs),
    'n_val': len(val_pairs),
    'n_test': len(test_pairs),
    'test_audit_id': test_hash,
    'corpus_sha256': corpus_sha,
}
audit_path = os.path.join(DATA_DIR, 'audit.json')
if os.path.exists(audit_path):
    with open(audit_path) as f:
        previous = json.load(f)
    assert previous == audit, f'La auditoría no coincide con la guardada en Drive:\n{previous}\nvs\n{audit}'
    print('Auditoría idéntica a la guardada previamente en Drive.')
else:
    with open(audit_path, 'w') as f:
        json.dump(audit, f, indent=2)
    print('Auditoría guardada en', audit_path)
pd.Series(audit, name='valor').to_frame()

Comparamos con la **partición oficial del equipo** (`EXPECTED_COUNTS` y `EXPECTED_TEST_AUDIT_ID` en `common/data.py`)
y guardamos un respaldo en Drive como `splits.json`. Los notebooks 02–06 cargan las particiones con
`get_splits()`, que usa ese respaldo y vuelve a verificar el Test Audit ID.

In [ ]:
from common.data import EXPECTED_COUNTS, EXPECTED_TEST_AUDIT_ID, save_splits_json, load_splits_json, get_splits

assert (len(train_pairs), len(val_pairs), len(test_pairs)) == EXPECTED_COUNTS
assert test_hash == EXPECTED_TEST_AUDIT_ID
print('OK: coincide con la partición oficial del equipo:', EXPECTED_COUNTS, EXPECTED_TEST_AUDIT_ID)

splits_path = os.path.join(DATA_DIR, 'splits.json')
if os.path.exists(splits_path):
    tr, va, te, h = load_splits_json(splits_path)
    assert tr == train_pairs and va == val_pairs and te == test_pairs
    print('splits.json en Drive es idéntico a la partición recién generada.')
else:
    save_splits_json(splits_path, train_pairs, val_pairs, test_pairs)
_ = get_splits(DATA_DIR)

## 3. Exploración de los datos

In [ ]:
pd.set_option('display.max_colwidth', 120)
pd.DataFrame(train_pairs[:5], columns=['en', 'es'])

In [ ]:
def length_table(pairs, name):
    return pd.DataFrame({
        'split': name,
        'palabras_en': [len(en.split()) for en, es in pairs],
        'palabras_es': [len(es.split()) for en, es in pairs],
    })

lengths = pd.concat([length_table(train_pairs, 'train'),
                     length_table(val_pairs, 'val'),
                     length_table(test_pairs, 'test')])
lengths.groupby('split')[['palabras_en', 'palabras_es']].describe().round(1).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ['palabras_en', 'palabras_es'], ['Inglés (fuente)', 'Español (destino)']):
    for split in ['train', 'val', 'test']:
        ax.hist(lengths.loc[lengths.split == split, col], bins=range(0, 61), alpha=0.5,
                density=True, label=split)
    ax.axvline(18.5, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'Longitud en palabras — {title}')
    ax.set_xlabel('palabras por oración')
    ax.set_ylabel('proporción')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'distribucion_longitudes.png'), dpi=150)
plt.show()

**Grupos de longitud para la Tabla 1.** Medimos la longitud en palabras de la oración **fuente en inglés**
(mismo criterio `en.split()` que usa el filtro del protocolo):
*media* = 8–18 palabras, *larga* = más de 18. Como el filtro exige al menos 8 palabras, todo par cae en uno de los dos grupos.

In [ ]:
from common.data import length_bucket

buckets = pd.DataFrame({
    split: pd.Series([length_bucket(en) for en, es in pairs]).value_counts()
    for split, pairs in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]
}).reindex(['media', 'larga'])
buckets.loc['% larga'] = (100 * buckets.loc['larga'] / buckets.sum()).round(2)
buckets

## 4. Auditoría de solapamiento entre particiones

Tatoeba contiene varias traducciones distintas de una misma oración en inglés. Como el protocolo baraja **pares**
(no oraciones), la misma frase en inglés puede quedar en train y también en test con otra traducción.
No es una fuga que nosotros introduzcamos (el protocolo es obligatorio), pero conviene **medirla y reportarla**
porque ayuda a interpretar el BLEU en test.

*Nota de integridad:* aquí solo se **cuenta** texto de test; no se usa para entrenar ni para elegir hiperparámetros.

In [ ]:
train_en = set(en for en, es in train_pairs)
overlap_val = sum(en in train_en for en, es in val_pairs)
overlap_test = sum(en in train_en for en, es in test_pairs)
pd.DataFrame({
    'pares': [len(val_pairs), len(test_pairs)],
    'inglés idéntico a uno de train': [overlap_val, overlap_test],
    '%': [round(100 * overlap_val / len(val_pairs), 2), round(100 * overlap_test / len(test_pairs), 2)],
}, index=['val', 'test'])

## 5. Preprocesamiento

La función `clean` (en `common/preprocess.py`) aplica: minúsculas → quitar acentos → quitar signos de puntuación.
Se usa para:
* preparar la entrada y salida de los modelos desde cero (A, B, C), y
* normalizar **predicciones y referencias de los 4 modelos** antes de calcular BLEU (Regla 3: normalización simétrica).

In [ ]:
from common.preprocess import clean

for en, es in train_pairs[:3]:
    print('EN :', en)
    print('   ->', clean(en))
    print('ES :', es)
    print('   ->', clean(es))
    print()

## 6. Vocabularios (solo con `train_pairs`)

Seguimos la **Guía de Implementación** de las slides *Transformer Parte 2* (`build_vocab`):
cada oración limpia se envuelve con `<sos> ... <eos>`, se tokeniza con `.split()`, se cuenta con `collections.Counter`
y se ordena de mayor a menor frecuencia. Índices: `<pad>=0` (coincide con `padding_idx=0` y con las máscaras),
`<unk>=1`, y el resto (incluidos `<sos>` y `<eos>`) desde 2.

Las palabras de val/test que no estén en el vocabulario de train se convierten en `<unk>` (Regla 1: cero fuga de datos).

Además probamos un umbral mínimo de frecuencia (`count_threshold`), como el vocabulario cerrado del notebook de N-gramas.
Lo elegimos mirando **solo validación**: tamaño del vocabulario vs. porcentaje de palabras `<unk>`.

In [ ]:
from common.vocab import build_vocab, oov_rate

train_en = [clean(en) for en, es in train_pairs]
train_es = [clean(es) for en, es in train_pairs]
val_en = [clean(en) for en, es in val_pairs]
val_es = [clean(es) for en, es in val_pairs]

rows = []
for count_threshold in [1, 2, 3, 5]:
    e2i = build_vocab(train_en, count_threshold)
    s2i = build_vocab(train_es, count_threshold)
    rows.append({
        'count_threshold': count_threshold,
        'vocab_en': len(e2i),
        'vocab_es': len(s2i),
        '% <unk> val (en)': round(oov_rate(val_en, e2i), 2),
        '% <unk> val (es)': round(oov_rate(val_es, s2i), 2),
        '% <unk> train (es)': round(oov_rate(train_es, s2i), 2),
    })
pd.DataFrame(rows).set_index('count_threshold')

**Decisión:** `count_threshold = 2`.

* Con `1` (la guía de la clase sin umbral), ninguna palabra de train es `<unk>`: el modelo nunca aprende qué hacer con `<unk>`,
  aunque en val/test sí aparece. Además, la capa de salida (softmax sobre el vocabulario español) es más grande.
* Con `2`, las palabras que aparecen una sola vez en train se vuelven `<unk>` (como en el notebook de N-gramas)
  y el vocabulario se reduce, a cambio de un aumento de `<unk>` en validación que debe ser pequeño (ver tabla).
* Valores mayores aumentan demasiado los `<unk>`, que se pagan directamente en BLEU.

In [ ]:
from common.vocab import save_vocab, invert, sentence_to_ids, ids_to_sentence

COUNT_THRESHOLD = 2
eng_word2idx = build_vocab(train_en, COUNT_THRESHOLD)
spa_word2idx = build_vocab(train_es, COUNT_THRESHOLD)
eng_idx2word, spa_idx2word = invert(eng_word2idx), invert(spa_word2idx)
print(f'Vocabulario inglés : {len(eng_word2idx):,} palabras')
print(f'Vocabulario español: {len(spa_word2idx):,} palabras')
print('Primeras entradas (es):', list(spa_word2idx.items())[:10])
print("<sos> =", spa_word2idx['<sos>'], "| <eos> =", spa_word2idx['<eos>'])

save_vocab(os.path.join(DATA_DIR, 'vocab.json'), eng_word2idx, spa_word2idx, COUNT_THRESHOLD)

**Verificación de cero fuga:** todas las palabras del vocabulario (salvo `<pad>` y `<unk>`) aparecen en train.

In [ ]:
from common.vocab import UNK_IDX, PAD_TOKEN, UNK_TOKEN, wrap

train_en_words = set(w for s in train_en for w in wrap(s))
train_es_words = set(w for s in train_es for w in wrap(s))
assert set(eng_word2idx) - {PAD_TOKEN, UNK_TOKEN} <= train_en_words
assert set(spa_word2idx) - {PAD_TOKEN, UNK_TOKEN} <= train_es_words
print('OK: los vocabularios se construyeron únicamente con train_pairs.')

# Ejemplo: una oración de validación con palabras desconocidas
for es in val_es:
    ids = sentence_to_ids(es, spa_word2idx)
    if UNK_IDX in ids:
        print('ES limpio        :', es)
        print('ids              :', ids)
        print('de vuelta a texto:', ids_to_sentence(ids, spa_idx2word))
        break

Comprobamos cuántas oraciones superan `MAX_SEQ_LEN = 128` (incluyendo `<sos>` y `<eos>`).
Siguiendo la guía de la clase, `collate_fn` trunca a ese largo; aquí verificamos que casi nunca ocurre.

In [ ]:
from common.datasets import pairs_to_ids, MAX_SEQ_LEN

train_src, train_tgt = pairs_to_ids(train_pairs, eng_word2idx, spa_word2idx)
val_src, val_tgt = pairs_to_ids(val_pairs, eng_word2idx, spa_word2idx)
print('MAX_SEQ_LEN =', MAX_SEQ_LEN)
for name, src, tgt in [('train', train_src, train_tgt), ('val', val_src, val_tgt)]:
    n_long = sum(len(s) > MAX_SEQ_LEN or len(t) > MAX_SEQ_LEN for s, t in zip(src, tgt))
    print(f'{name:5s}: más larga fuente {max(map(len, src))} | destino {max(map(len, tgt))} tokens'
          f' | pares truncados: {n_long}')

## 7. Resumen para el reporte

In [ ]:
summary = {
    **audit,
    'test_media (8-18)': int(buckets.loc['media', 'test']),
    'test_larga (>18)': int(buckets.loc['larga', 'test']),
    'count_threshold': COUNT_THRESHOLD,
    'vocab_en': len(eng_word2idx),
    'vocab_es': len(spa_word2idx),
    'test con inglés visto en train (%)': round(100 * overlap_test / len(test_pairs), 2),
}
with open(os.path.join(DATA_DIR, 'summary_01.json'), 'w') as f:
    json.dump(summary, f, indent=2)
pd.Series(summary, name='valor').to_frame()